In [1]:
import numpy as np
import torch
from scipy import stats
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

import sys
sys.path.append('../../src')
root = Path('../../src')
from utils.yaml import parse_yaml
from utils.utils import load_checkpoint
from data.transforms import numpy_to_floattensor, numpy_to_doubletensor
from metrics.testing import hsic, mmd
from kernel.gaussian import Gaussian
device = torch.device('cuda:0')

In [ ]:
# load dataset
dataset_cfg = root/"config/dataset/sinusoid/sinusoid-f1-generator.yml"
dataconfig = parse_yaml(dataset_cfg)['test']
if 'root' in dataconfig:
    dataconfig['root'] = root/dataconfig['root']
dataconfig['transform'] = numpy_to_floattensor
dataset = dataconfig.build()

def stream(dataloader):
    while True:
        yield from dataloader

In [ ]:
BANDWIDTH_X = BANDWIDTH_Y = 1.
NPERM = 200
NTEST = 500
MMD_NPERMS = [1,2,3,5,10,20,30,40,50] #[1, 5, 10, 20] #, 30, 40, 50]

In [ ]:
def mT(X: np.ndarray):
    return X.swapaxes(-2,-1)

def dist2(X: np.ndarray, Y: np.ndarray):
    assert X.ndim >= 2 and Y.ndim >= 2
    xnorm2 = np.sum(X**2, axis=-1, keepdims=True)
    ynorm2 = np.sum(Y**2, axis=-1, keepdims=True)
    return np.clip(xnorm2 - 2 * X @ mT(Y) + mT(ynorm2), a_min=0, a_max=None)

def k(X, Y, bandwidth=BANDWIDTH_X):
    Dxy2 = dist2(X, Y)  # (nx, ny)
    return np.exp(-0.5*Dxy2/bandwidth**2)

def l(X, Y, bandwidth=BANDWIDTH_Y):
    Dxy2 = dist2(X, Y)  # (nx, ny)
    return np.exp(-0.5*Dxy2/bandwidth**2)


def mmd2_b(X, Y, p, k, l):
    # p is (p,n) array of permutations
    pY = Y[p]               # (p,n,d)
    _pY = pY[:,np.newaxis]  # for broadcasting
    Kxx = k(X, X)           # (n, n)
    Lyy = l(Y, Y)           # (n, n)
    Lypy = l(Y, pY)         # (p, n, n)
    Lpypy = l(pY, _pY)      # (p, p, n, n)
    s1 = np.mean(Kxx * Lyy)
    s2 = np.mean(Kxx * Lypy)
    s3 = np.mean(Kxx * Lpypy)
    return s1 - 2*s2 + s3

def hsic_b(X, Y, k, l):
    n = X.shape[0]
    Kxx = k(X, X)
    Lyy = l(Y, Y)
    s1 = np.mean(Kxx * Lyy)
    s2 = np.einsum('ij,iq->', Kxx, Lyy) / (n**3)
    s3 = np.mean(Kxx) * np.mean(Lyy)
    return s1 - 2*s2 + s3

In [ ]:
dataloader = DataLoader(dataset, batch_size=50)

stats = defaultdict(lambda: np.empty(NTEST))
threshs = defaultdict(lambda: np.empty(NTEST))

for i, batch in tqdm(zip(range(NTEST), dataloader), total=NTEST):

    X, Y = batch
    X = X.numpy()
    Y = Y.numpy()
    n = X.shape[0]
    # Y = Y[np.random.permutation(n)]

    perms = np.array([np.random.permutation(n) for _ in range(max(MMD_NPERMS))])
    stats['hsic'][i] = hsic_b(X, Y, k, l)
    for p in MMD_NPERMS:
        stats[f'mmd2,p={p}'][i] = mmd2_b(X, Y, perms[:p], k, l)


    # perm test
    nulls = defaultdict(lambda: np.empty(NPERM))
    for j in range(NPERM):
        Y0 = Y[np.random.permutation(n)]
        nulls['hsic'][j] = hsic_b(X, Y0, k, l)
        perms = np.array([np.random.permutation(n) for _ in range(max(MMD_NPERMS))])
        for p in MMD_NPERMS:
            nulls[f'mmd2,p={p}'][j] = mmd2_b(X, Y0, perms[:p], k, l)

    threshs['hsic'][i] = np.quantile(nulls['hsic'], q=0.95, method='inverted_cdf')
    for p in MMD_NPERMS:
        threshs[f'mmd2,p={p}'][i] = np.quantile(nulls[f'mmd2,p={p}'], q=0.95, method='inverted_cdf')

power = dict()
power['hsic'] = (stats['hsic'] > threshs['hsic']).mean()
for p in MMD_NPERMS:
    power[f'mmd2,p={p}'] = (stats[f'mmd2,p={p}'] > threshs[f'mmd2,p={p}']).mean()

In [ ]:
np.save('checkpoints/data-6', (dict(stats), dict(threshs)))

In [ ]:
stats, threshs = np.load('checkpoints/data-6.npy', allow_pickle=True)

In [ ]:
power = {'hsic': np.empty(len(MMD_NPERMS)), 'mmd2': np.empty(len(MMD_NPERMS))}
mean = {'hsic': np.empty(len(MMD_NPERMS)), 'mmd2': np.empty(len(MMD_NPERMS))}
var = {'hsic': np.empty(len(MMD_NPERMS)), 'mmd2': np.empty(len(MMD_NPERMS))}

power['hsic'][:] = (stats['hsic'] > threshs['hsic']).mean()
mean['hsic'][:] = stats['hsic'].mean()
var['hsic'][:] = stats['hsic'].var()

for i,p in enumerate(MMD_NPERMS):
    power['mmd2'][i] = (stats[f'mmd2,p={p}'] > threshs[f'mmd2,p={p}']).mean()
    mean['mmd2'][i] = stats[f'mmd2,p={p}'].mean()
    var['mmd2'][i] = stats[f'mmd2,p={p}'].var()

print(f"{power=}")
print(f"{mean=}")
print(f"{var=}")

In [ ]:
fig, ax = plt.subplots(squeeze=True, layout='constrained') #, figsize=(12, 5),)

ax.plot(MMD_NPERMS, var['mmd2'], marker='o', ms=10, linewidth=2, color='C0', label=r'$\widehat{\text{MMD}}^2(\mathbb{Z}, \sigma\mathbb{Z})$')
ax.plot(MMD_NPERMS, var['hsic'], linestyle='dashed', ms=10, linewidth=3, color='C2', label=r'$\widehat{\text{HSIC}}(\mathbb{Z}) $')
ax.set_title("")
ax.set_xlabel(r'# permutations $p$', fontsize=24)
ax.set_ylabel(r'variance', fontsize=24)
ax.legend(fontsize=16)

# axis properties
ax.grid(linewidth=0.2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1e'))
ax.tick_params(left=False, bottom=False, labelsize=14)
# ax.set_yticklabels([])
# ax.set_xticklabels([])
# ax.set_aspect('equal')
# ax.set_box_aspect(1)

# plt.savefig(f'var-vs-p.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(squeeze=True, layout='constrained') #, figsize=(12, 5),)

ax.plot(MMD_NPERMS, power['mmd2'], marker='o', ms=10, linewidth=2, color='C0', label=r'$\widehat{\text{MMD}}^2(\mathbb{Z}, \sigma\mathbb{Z})$')
ax.plot(MMD_NPERMS, power['hsic'], linestyle='dashed', ms=10, linewidth=3, color='C2', label=r'$\widehat{\text{HSIC}}(\mathbb{Z}) $')
ax.set_title("")
ax.set_xlabel(r'# permutations $p$', fontsize=24)
ax.set_ylabel(r'power', fontsize=24)
ax.legend(fontsize=16)

import matplotlib.ticker as ticker

# axis properties
ax.grid(linewidth=0.2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.tick_params(left=False, bottom=False, labelsize=14)
# ax.set_yticklabels([])
# ax.set_xticklabels([])
# ax.set_aspect('equal')
# ax.set_box_aspect(1)

# plt.savefig(f'power-vs-p.pdf', bbox_inches='tight')

In [ ]:
dataloader = DataLoader(dataset, batch_size=50)

# stats = {'mmd2': np.empty(NTEST), 'hsic': np.empty(NTEST)}
# threshs = {'mmd2': np.empty(NTEST), 'hsic': np.empty(NTEST)}

stats = defaultdict(lambda: np.empty(NTEST))
threshs = defaultdict(lambda: np.empty(NTEST))

for i, batch in tqdm(zip(range(NTEST), dataloader), total=NTEST):

    X, Y = batch
    X = X.numpy()
    Y = Y.numpy()
    n = X.shape[0]
    # Y = Y[np.random.permutation(n)]

    p = np.array([np.random.permutation(n) for _ in range(20)])
    stats['mmd2'][i] = mmd2_b(X, Y, p, k, l)
    stats['hsic'][i] = hsic_b(X, Y, k, l)


    # perm test
    nulls = {'mmd2': np.empty(NPERM), 'hsic': np.empty(NPERM)}
    for j in range(NPERM):
        Y0 = Y[np.random.permutation(n)]
        p = np.array([np.random.permutation(n) for _ in range(20)])
        nulls['mmd2'][j] = mmd2_b(X, Y0, p, k, l)
        nulls['hsic'][j] = hsic_b(X, Y0, k, l)

    threshs['mmd2'][i] = np.quantile(nulls['mmd2'], q=0.95, method='inverted_cdf')
    threshs['hsic'][i] = np.quantile(nulls['hsic'], q=0.95, method='inverted_cdf')

power = dict()
power['mmd2'] = (stats['mmd2'] > threshs['mmd2']).mean()
power['hsic'] = (stats['hsic'] > threshs['hsic']).mean()

In [ ]:
RUNS = [2,3,4,5]
hsic_vars = np.empty(len(RUNS))
hsic_pows = np.empty(len(RUNS))
mmd2_vars = defaultdict(lambda: np.empty(len(RUNS)))
mmd2_pows = defaultdict(lambda: np.empty(len(RUNS)))

for i,r in enumerate(RUNS):
    stats, threshs = np.load(f'data-{r}.npy', allow_pickle=True)

    hsic_vars[i] = stats['hsic'].var()
    hsic_pows[i] = (stats['hsic'] > threshs['hsic']).mean()

    for p in MMD_NPERMS:
        mmd2_vars[f"p={p}"][i] = stats[f'mmd2,p={p}'].var()
        mmd2_pows[f"p={p}"][i] = (stats[f'mmd2,p={p}'] > threshs[f'mmd2,p={p}']).mean()


# compute statistics per permutation p
var = defaultdict(lambda: np.empty(len(MMD_NPERMS)))
vse = defaultdict(lambda: np.empty(len(MMD_NPERMS)))    # standard error
power = defaultdict(lambda: np.empty(len(MMD_NPERMS)))
powse = defaultdict(lambda: np.empty(len(MMD_NPERMS)))  # standard error

var['hsic'][:] = hsic_vars.mean()
vse['hsic'][:] = hsic_vars.std()
power['hsic'][:] = hsic_pows.mean()
powse['hsic'][:] = hsic_pows.std()
for i,p in enumerate(MMD_NPERMS):
    var['mmd2'][i] = mmd2_vars[f"p={p}"].mean()
    vse['mmd2'][i] = mmd2_vars[f"p={p}"].std()
    power['mmd2'][i] = mmd2_pows[f"p={p}"].mean()
    powse['mmd2'][i] = mmd2_pows[f"p={p}"].std()

In [ ]:
fig, ax = plt.subplots(squeeze=True, layout='constrained') #, figsize=(12, 5),)

ax.plot(MMD_NPERMS, var['mmd2'], marker='o', ms=10, linewidth=2, color='C0', label=r'$\widehat{\text{MMD}}^2(\mathbb{Z}, \sigma\mathbb{Z})$')
ax.plot(MMD_NPERMS, var['hsic'], linestyle='dashed', ms=10, linewidth=3, color='C2', label=r'$\widehat{\text{HSIC}}(\mathbb{Z}) $')

ax.fill_between(MMD_NPERMS, var['mmd2']-vse['mmd2'], var['mmd2']+vse['mmd2'], alpha=0.2, color='C0')
ax.fill_between(MMD_NPERMS, var['hsic']-vse['hsic'], var['hsic']+vse['hsic'], alpha=0.2, color='C2')

ax.set_title("")
ax.set_xlabel(r'# permutations $p$', fontsize=24)
ax.set_ylabel(r'variance', fontsize=24)
ax.legend(fontsize=20)

# axis properties
ax.grid(linewidth=0.2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1e'))
ax.tick_params(left=False, bottom=False, labelsize=14)
# ax.set_yticklabels([])
# ax.set_xticklabels([])
# ax.set_aspect('equal')
# ax.set_box_aspect(1)

# plt.savefig(f'var-vs-p.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(squeeze=True, layout='constrained') #, figsize=(12, 5),)

ax.plot(MMD_NPERMS, power['mmd2'], marker='o', ms=10, linewidth=2, color='C0', label=r'$\widehat{\text{MMD}}^2(\mathbb{Z}, \sigma\mathbb{Z})$')
ax.plot(MMD_NPERMS, power['hsic'], linestyle='dashed', ms=10, linewidth=3, color='C2', label=r'$\widehat{\text{HSIC}}(\mathbb{Z}) $')

ax.fill_between(MMD_NPERMS, power['mmd2']-powse['mmd2'], power['mmd2']+powse['mmd2'], alpha=0.2, color='C0')
ax.fill_between(MMD_NPERMS, power['hsic']-powse['hsic'], power['hsic']+powse['hsic'], alpha=0.2, color='C2')


ax.set_title("")
ax.set_xlabel(r'# permutations $p$', fontsize=24)
ax.set_ylabel(r'power', fontsize=24)
ax.legend(fontsize=20)

# axis properties
ax.grid(linewidth=0.2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.tick_params(left=False, bottom=False, labelsize=14)
# ax.set_yticklabels([])
# ax.set_xticklabels([])
# ax.set_aspect('equal')
# ax.set_box_aspect(1)

# plt.savefig(f'power-vs-p.pdf', bbox_inches='tight')